<a href="https://colab.research.google.com/github/IgorTTenorio/ai-python-playground/blob/main/ai_python_challenge2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Desafio:
1. Criar um arquivo .txt a partir de uma lista de perguntas vindas de uma lista em Python
2. Ler as perguntas desse arquivo .txt
3. Obter as respostas de um LLM para cada uma
4. Salvar os resultados em um novo arquivo .csv
5. Ler o arquivo .csv usando Pandas

In [7]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.4 MB/s eta 0:00:00


In [9]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [1]:
# Etapa 1
lista_perguntas = [
    "Do que é feito o Sol?",
    "Do que é feito os aneis de Saturno?",
    "Quantos anos luz está a terra de Júpiter?",
    "Qual é a estrela mais proxima do Sol?",
    "Qual é o tamanho de plutão?"
]

with open('lista_perguntas.txt', 'w', encoding='utf-8') as arquivo:
  for pergunta in lista_perguntas:
    arquivo.write(pergunta + "\n")

In [13]:
# Etapa 2
nova_lista_perguntas = []

with open('lista_perguntas.txt', 'r', encoding='utf-8') as arquivo:
  for linha in arquivo:
    nova_lista_perguntas.append(linha.strip())

In [11]:
# Etapa 3
from groq import Groq

client = Groq()

In [60]:
def provedor_de_respostas(perguntas):
  respostas = []
  for numero, pergunta in enumerate(perguntas):
    resposta = ""
    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
          {
            "role": "user",
            "content": pergunta
          }
        ],
        temperature=1,
        max_completion_tokens=8192,
        top_p=1,
        reasoning_effort="medium",
        stream=True,
        stop=None
    )
    for chunk in completion:
      content = chunk.choices[0].delta.content
      if content:
          resposta += content
    respostas.append({"ID": str(numero+1), "Pergunta": pergunta, "Resposta": resposta})
  return respostas


In [61]:
lista_respostas = provedor_de_respostas(nova_lista_perguntas)

In [62]:
for resposta in lista_respostas:
  print(resposta)

{'ID': '1', 'Pergunta': 'Do que é feito o Sol?', 'Resposta': '**O Sol é uma estrela de tipo espectral G2\u202fV, e sua composição química reflete, em grande parte, a da nebulosa solar que deu origem ao Sistema Solar.**  \n\n| Elemento | Fração de massa | Fração de número de átomos |\n|----------|----------------|----------------------------|\n| Hidrogênio (H) | ≈\u202f74\u202f% | ≈\u202f91\u202f% |\n| Hélio (He) | ≈\u202f24\u202f% | ≈\u202f8,9\u202f% |\n| “Metais” (todos os demais, como O, C, N, Ne, Fe, Si, Mg, etc.) | ≈\u202f2\u202f% | ≈\u202f0,1\u202f% |\n\n### Principais elementos “pesados” (metais)\n- **Oxigênio (O)** – ~0,8\u202f% da massa solar  \n- **Carbono (C)** – ~0,3\u202f%  \n- **Nitrogênio (N)** – ~0,1\u202f%  \n- **Neônio (Ne)** – ~0,1\u202f%  \n- **Ferro (Fe)** – ~0,1\u202f%  \n- **Silício (Si), magnésio (Mg), enxofre (S)** e outros completam o restante do ~2\u202f%.\n\n\n\n## Como esses elementos estão distribuídos nas camadas solares\n\n| Camada | Características princ

In [63]:
# Etapa 4
with open('lista_respostas.csv', 'w', encoding='utf-8') as arquivo_csv:
  arquivo_csv.write(';'.join(("ID","Pergunta","Resposta")) + '\n')
  for resposta in lista_respostas:
    arquivo_csv.write(';'.join((resposta.get("ID"), resposta.get("Pergunta"), resposta.get("Resposta"))) + '\n')

In [64]:
import csv

with open("lista_respostas.csv", "w", newline="", encoding="utf-8") as arquivo_csv:
    writer = csv.writer(arquivo_csv)
    writer.writerow(["ID", "Pergunta","Resposta"])
    for resposta in lista_respostas:
      writer.writerow([resposta.get("ID"), resposta.get("Pergunta"), resposta.get("Resposta")])

In [65]:
# Etapa 5
import pandas as pd
df = pd.read_csv("/content/lista_respostas.csv")

In [66]:
df

,ID,Pergunta,Resposta
0,1,Do que é feito o Sol?,"**O Sol é uma estrela de tipo espectral G2 V, ..."
1,2,Do que é feito os aneis de Saturno?,Os anéis de Saturno são formados basicamente p...
2,3,Quantos anos luz está a terra de Júpiter?,**Distância Terra‑Júpiter em anos‑luz**\n\nA d...
3,4,Qual é a estrela mais proxima do Sol?,A estrela mais próxima do Sol é **Próxima Cent...
4,5,Qual é o tamanho de plutão?,Plutão é um planeta anão localizado na região ...
